# Intraday Momentum via Top Movers on MOEX

Test whether stocks with the largest first-hour returns (10:00-11:00 MSK) exhibit continuation or reversal for the rest of the trading day. Part 1 uses top-20 liquid stocks, Part 2 stress-tests on ~100 stocks with beta-neutral and volume filters.

In [ ]:
# Setup and universe definition
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime, timedelta
from scipy import stats
import time
import warnings
warnings.filterwarnings("ignore")

plt.rcParams["figure.figsize"] = (14, 5)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

MOEX_BASE = "https://iss.moex.com/iss"

LIQUID_STOCKS = [
    "SBER", "SBERP", "GAZP", "LKOH", "ROSN",
    "NVTK", "GMKN", "MGNT", "MOEX", "VTBR",
    "TATN", "PLZL", "CHMF", "NLMK", "MTSS",
    "SNGS", "AFLT", "ALRS", "PHOR", "RUAL",
]

# Trader tariff costs
COMMISSION_PCT = 0.0005   # 0.05% per trade
SLIPPAGE_PCT = 0.0008     # ~0.08% slippage on liquid top-20
TOTAL_COST_PCT = (COMMISSION_PCT + SLIPPAGE_PCT) * 2  # round trip


def moex_get_stock_candles(security, interval=60, start="2024-01-01", end=None,
                            timeout=60, max_retries=3):
    if end is None:
        end = datetime.now().strftime("%Y-%m-%d")
    all_rows, page_start = [], 0
    while True:
        url = f"{MOEX_BASE}/engines/stock/markets/shares/securities/{security}/candles.json"
        params = {"from": start, "till": end, "interval": interval, "start": page_start}
        resp = None
        for attempt in range(max_retries):
            try:
                resp = requests.get(url, params=params, timeout=timeout)
                resp.raise_for_status()
                break
            except Exception as e:
                if attempt == max_retries - 1:
                    print(f"  Error {security} after {max_retries} attempts: {e}")
                    return pd.DataFrame()
                time.sleep(2 ** attempt)
        if resp is None:
            break
        candles = resp.json()["candles"]
        rows = candles["data"]
        if not rows:
            break
        all_rows.extend(rows)
        page_start += len(rows)
        time.sleep(0.2)
    if not all_rows:
        return pd.DataFrame()
    df = pd.DataFrame(all_rows, columns=candles["columns"])
    df["begin"] = pd.to_datetime(df["begin"])
    df = df.rename(columns={"begin": "timestamp", "value": "volume_rub", "volume": "vol"})
    df["ticker"] = security
    return df.sort_values("timestamp").reset_index(drop=True)


print(f"Universe: {len(LIQUID_STOCKS)} stocks")
print(f"Commission (Trader): {COMMISSION_PCT*100:.3f}% per trade")
print(f"Slippage:            {SLIPPAGE_PCT*100:.3f}% per trade")
print(f"Total RT cost:       {TOTAL_COST_PCT*100:.3f}%")

In [ ]:
# Data load: hourly candles for all 20 stocks
START_DATE = "2024-01-01"

frames = []
for ticker in LIQUID_STOCKS:
    print(f"  {ticker}...", end=" ")
    df = moex_get_stock_candles(ticker, interval=60, start=START_DATE)
    if not df.empty:
        frames.append(df)
        print(f"{len(df)} candles, {df['timestamp'].min().date()} - {df['timestamp'].max().date()}")
    else:
        print("EMPTY")

stocks = pd.concat(frames, ignore_index=True)
stocks["date"] = stocks["timestamp"].dt.date
stocks["hour"] = stocks["timestamp"].dt.hour

print(f"\nTotal: {len(stocks)} candles, {stocks['ticker'].nunique()} stocks")
print(f"Period: {stocks['timestamp'].min().date()} - {stocks['timestamp'].max().date()}")
print(f"Trading days: {stocks['date'].nunique()}")

In [ ]:
# Extract intraday sessions: first hour (10-11) vs rest of day (11-18:45)
def extract_intraday_sessions(df):
    sessions = []
    for (ticker, d), grp in df.groupby(["ticker", "date"]):
        grp = grp.sort_values("timestamp")
        main = grp[(grp["hour"] >= 10) & (grp["hour"] <= 18)]
        if len(main) < 3:
            continue
        first_hour = main[main["hour"] == 10]
        if first_hour.empty:
            continue
        open_10 = first_hour.iloc[0]["open"]
        close_11 = first_hour.iloc[0]["close"]
        vol_first = first_hour.iloc[0]["volume_rub"]
        rest = main[main["hour"] > 10]
        if rest.empty:
            continue
        close_eod = rest.iloc[-1]["close"]
        high_rest = rest["high"].max()
        low_rest = rest["low"].min()
        first_hour_ret = (close_11 - open_10) / open_10
        rest_of_day_ret = (close_eod - close_11) / close_11
        full_day_ret = (close_eod - open_10) / open_10
        sessions.append({
            "ticker": ticker, "date": d,
            "open_10": open_10, "close_11": close_11, "close_eod": close_eod,
            "high_rest": high_rest, "low_rest": low_rest,
            "vol_first_hour": vol_first,
            "first_hour_ret": first_hour_ret,
            "rest_of_day_ret": rest_of_day_ret,
            "full_day_ret": full_day_ret,
        })
    return pd.DataFrame(sessions)


sessions = extract_intraday_sessions(stocks)
sessions["first_hour_ret_pct"] = sessions["first_hour_ret"] * 100
sessions["rest_of_day_ret_pct"] = sessions["rest_of_day_ret"] * 100

print(f"Sessions: {len(sessions)}")
print(f"\nfirst_hour_ret stats:")
print(sessions["first_hour_ret_pct"].describe().round(2))
print(f"\nrest_of_day_ret stats:")
print(sessions["rest_of_day_ret_pct"].describe().round(2))

In [ ]:
# Correlation test: first hour return vs rest of day
corr_all, p_all = stats.pearsonr(sessions["first_hour_ret"], sessions["rest_of_day_ret"])
print(f"Overall correlation first_hour_ret vs rest_of_day_ret: {corr_all:+.4f} (p={p_all:.4f})")

print(f"\nCorrelation by ticker:")
ticker_corrs = []
for t, grp in sessions.groupby("ticker"):
    if len(grp) < 30:
        continue
    c, p = stats.pearsonr(grp["first_hour_ret"], grp["rest_of_day_ret"])
    ticker_corrs.append({"ticker": t, "n": len(grp), "corr": c, "p_value": p})
ticker_corrs_df = pd.DataFrame(ticker_corrs).sort_values("corr")
display(ticker_corrs_df.round(4))

print(f"\nCorrelation by first-hour return quintile:")
sessions["fh_quintile"] = pd.qcut(sessions["first_hour_ret"], 5,
                                   labels=["Q1 (worst)", "Q2", "Q3", "Q4", "Q5 (best)"])

quintile_stats = sessions.groupby("fh_quintile").agg(
    n=("rest_of_day_ret", "count"),
    fh_mean=("first_hour_ret_pct", "mean"),
    rod_mean=("rest_of_day_ret_pct", "mean"),
    rod_median=("rest_of_day_ret_pct", "median"),
    rod_std=("rest_of_day_ret_pct", "std"),
    rod_win_rate=("rest_of_day_ret", lambda x: (x > 0).mean() * 100),
).round(3)
display(quintile_stats)

In [ ]:
# Scatter and binned scatter visualization
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

ax = axes[0]
ax.scatter(sessions["first_hour_ret_pct"], sessions["rest_of_day_ret_pct"],
           alpha=0.15, s=8, color="steelblue")
ax.axhline(0, color="black", linewidth=0.5)
ax.axvline(0, color="black", linewidth=0.5)
ax.set_xlabel("First hour return, %")
ax.set_ylabel("Rest of day return, %")
ax.set_title(f"Scatter: first hour vs rest of day (n={len(sessions)})\nCorr = {corr_all:+.3f}")

slope, intercept, r_value, p_value, _ = stats.linregress(
    sessions["first_hour_ret_pct"], sessions["rest_of_day_ret_pct"])
x_range = np.linspace(sessions["first_hour_ret_pct"].min(),
                       sessions["first_hour_ret_pct"].max(), 100)
ax.plot(x_range, slope * x_range + intercept, color="red", linewidth=2,
        label=f"y = {slope:.3f}x + {intercept:.3f}")
ax.legend()

ax = axes[1]
sessions["fh_bin"] = pd.qcut(sessions["first_hour_ret"], 20, labels=False, duplicates="drop")
bin_stats = sessions.groupby("fh_bin").agg(
    fh_mean=("first_hour_ret_pct", "mean"),
    rod_mean=("rest_of_day_ret_pct", "mean"),
    rod_sem=("rest_of_day_ret_pct", lambda x: x.std() / np.sqrt(len(x))),
)

ax.errorbar(bin_stats["fh_mean"], bin_stats["rod_mean"], yerr=bin_stats["rod_sem"],
            fmt="o", color="steelblue", capsize=3, markersize=8)
ax.axhline(0, color="black", linewidth=0.5)
ax.axvline(0, color="black", linewidth=0.5)
ax.axhline(TOTAL_COST_PCT * 100, color="red", linestyle="--", alpha=0.5,
           label=f"RT cost = {TOTAL_COST_PCT*100:.2f}%")
ax.axhline(-TOTAL_COST_PCT * 100, color="red", linestyle="--", alpha=0.5)
ax.set_xlabel("First hour return, % (bin mean)")
ax.set_ylabel("Rest of day return, %")
ax.set_title("Binned: 20 quantiles by morning return")
ax.legend()

plt.tight_layout()
plt.show()

print(f"\nLinear regression: slope = {slope:.4f}, R2 = {r_value**2:.4f}, p = {p_value:.4f}")

In [ ]:
# Backtest: top-N movers strategy with cost accounting
def backtest_top_movers(sessions_df, top_n=3, side="long",
                         min_fh_ret_pct=None, cost_pct=TOTAL_COST_PCT):
    trades = []
    for d, day_sessions in sessions_df.groupby("date"):
        if min_fh_ret_pct is not None:
            if side in ["long", "short"]:
                day_sessions = day_sessions[day_sessions["first_hour_ret_pct"].abs() >= min_fh_ret_pct]
            else:
                day_sessions = day_sessions[day_sessions["first_hour_ret_pct"] <= -min_fh_ret_pct]
        if len(day_sessions) < top_n:
            continue
        if side == "long":
            picks = day_sessions.nlargest(top_n, "first_hour_ret")
            direction = 1
        elif side == "short":
            picks = day_sessions.nlargest(top_n, "first_hour_ret")
            direction = -1
        elif side == "long_losers":
            picks = day_sessions.nsmallest(top_n, "first_hour_ret")
            direction = 1
        for _, row in picks.iterrows():
            pnl_gross = direction * row["rest_of_day_ret"]
            pnl_net = pnl_gross - cost_pct
            trades.append({
                "date": d, "ticker": row["ticker"],
                "first_hour_ret": row["first_hour_ret"],
                "rest_of_day_ret": row["rest_of_day_ret"],
                "direction": "long" if direction == 1 else "short",
                "pnl_gross": pnl_gross, "pnl_net": pnl_net,
            })
    return pd.DataFrame(trades)


def calc_metrics(trades_df, label=""):
    if trades_df.empty:
        return {"label": label, "trades": 0}
    n = len(trades_df)
    wins = (trades_df["pnl_net"] > 0).sum()
    daily = trades_df.groupby("date")["pnl_net"].mean()
    sharpe = daily.mean() / daily.std() * np.sqrt(252) if daily.std() > 0 else 0
    cum = daily.cumsum()
    max_dd = (cum.cummax() - cum).max()
    annual_ret = daily.mean() * 252
    return {
        "label": label, "trades": n, "days": len(daily),
        "win_rate_pct": wins / n * 100,
        "avg_pnl_pct": trades_df["pnl_net"].mean() * 100,
        "median_pnl_pct": trades_df["pnl_net"].median() * 100,
        "total_ret_pct": daily.sum() * 100,
        "annual_ret_pct": annual_ret * 100,
        "sharpe": sharpe, "max_dd_pct": max_dd * 100,
        "best_day_pct": daily.max() * 100,
        "worst_day_pct": daily.min() * 100,
    }


def print_metrics(m):
    print(f"\n{'='*60}")
    print(f"  {m['label']}")
    print(f"{'='*60}")
    if m['trades'] == 0:
        print("  No trades")
        return
    print(f"  Trades:          {m['trades']} ({m['days']} trading days)")
    print(f"  Win rate:        {m['win_rate_pct']:.1f}%")
    print(f"  Avg PnL/trade:   {m['avg_pnl_pct']:+.3f}%")
    print(f"  Median PnL:      {m['median_pnl_pct']:+.3f}%")
    print(f"  Total return:    {m['total_ret_pct']:+.1f}%")
    print(f"  Annualized:      {m['annual_ret_pct']:+.1f}%")
    print(f"  Sharpe (ann):    {m['sharpe']:.2f}")
    print(f"  Max DD:          {m['max_dd_pct']:.1f}%")
    print(f"  Best/Worst day:  {m['best_day_pct']:+.2f}% / {m['worst_day_pct']:+.2f}%")


trades_base = backtest_top_movers(sessions, top_n=3, side="long")
m_base = calc_metrics(trades_base, "Top-3 LONG winners (continuation, no filter)")
print_metrics(m_base)

trades_1pct = backtest_top_movers(sessions, top_n=3, side="long", min_fh_ret_pct=1.0)
m_1pct = calc_metrics(trades_1pct, "Top-3 LONG, morning return >= 1%")
print_metrics(m_1pct)

trades_2pct = backtest_top_movers(sessions, top_n=3, side="long", min_fh_ret_pct=2.0)
m_2pct = calc_metrics(trades_2pct, "Top-3 LONG, morning return >= 2%")
print_metrics(m_2pct)

trades_rev = backtest_top_movers(sessions, top_n=3, side="short", min_fh_ret_pct=2.0)
m_rev = calc_metrics(trades_rev, "Top-3 SHORT (reversal), morning return >= 2%")
print_metrics(m_rev)

trades_losers = backtest_top_movers(sessions, top_n=3, side="long_losers", min_fh_ret_pct=2.0)
m_losers = calc_metrics(trades_losers, "Top-3 LONG losers, morning drop >= 2%")
print_metrics(m_losers)

In [ ]:
# Grid search over all configurations by Sharpe
configs = []
for top_n in [1, 2, 3, 5]:
    for thresh in [None, 0.5, 1.0, 1.5, 2.0, 3.0]:
        for side in ["long", "short", "long_losers"]:
            trades = backtest_top_movers(sessions, top_n=top_n, side=side, min_fh_ret_pct=thresh)
            m = calc_metrics(trades, f"top_n={top_n}, side={side}, thresh={thresh}")
            if m["trades"] >= 30:
                m["top_n"] = top_n
                m["side"] = side
                m["thresh"] = thresh
                configs.append(m)

configs_df = pd.DataFrame(configs).sort_values("sharpe", ascending=False)
print("TOP-15 configurations by Sharpe:")
display(configs_df[["top_n", "side", "thresh", "trades", "win_rate_pct",
                     "avg_pnl_pct", "annual_ret_pct", "sharpe", "max_dd_pct"]].head(15).round(3))

print("\nWORST-5:")
display(configs_df[["top_n", "side", "thresh", "trades", "win_rate_pct",
                     "avg_pnl_pct", "annual_ret_pct", "sharpe", "max_dd_pct"]].tail(5).round(3))

In [ ]:
# Equity curves for best and worst configurations
def thresh_clean(v):
    if v is None or pd.isna(v):
        return None
    return float(v)


best = configs_df.iloc[0]
worst = configs_df.iloc[-1]

best_trades = backtest_top_movers(
    sessions, top_n=int(best["top_n"]), side=best["side"],
    min_fh_ret_pct=thresh_clean(best["thresh"])
)
worst_trades = backtest_top_movers(
    sessions, top_n=int(worst["top_n"]), side=worst["side"],
    min_fh_ret_pct=thresh_clean(worst["thresh"])
)

print(f"Best trades: {len(best_trades)}, Worst trades: {len(worst_trades)}")

fig, axes = plt.subplots(2, 1, figsize=(14, 9))

if not best_trades.empty:
    daily_best = best_trades.groupby("date")["pnl_net"].mean()
    cum_best = daily_best.cumsum() * 100
    axes[0].plot(pd.to_datetime(daily_best.index), cum_best, color="#2ecc71", linewidth=1.5,
                 label=f"BEST: top_n={int(best['top_n'])}, side={best['side']}, thresh={best['thresh']}")
    axes[0].fill_between(pd.to_datetime(daily_best.index), cum_best, alpha=0.15, color="#2ecc71")
    axes[0].axhline(0, color="black", linewidth=0.5)
    axes[0].set_title(f"Equity (best): {best['annual_ret_pct']:.1f}% ann, Sharpe {best['sharpe']:.2f}")
    axes[0].set_ylabel("Cumulative return, %")
    axes[0].legend()

if not worst_trades.empty:
    daily_worst = worst_trades.groupby("date")["pnl_net"].mean()
    cum_worst = daily_worst.cumsum() * 100
    axes[1].plot(pd.to_datetime(daily_worst.index), cum_worst, color="#e74c3c", linewidth=1.5,
                 label=f"WORST: top_n={int(worst['top_n'])}, side={worst['side']}, thresh={worst['thresh']}")
    axes[1].fill_between(pd.to_datetime(daily_worst.index), cum_worst, alpha=0.15, color="#e74c3c")
    axes[1].axhline(0, color="black", linewidth=0.5)
    axes[1].set_title(f"Equity (worst): {worst['annual_ret_pct']:.1f}% ann, Sharpe {worst['sharpe']:.2f}")
    axes[1].set_ylabel("Cumulative return, %")
    axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Robustness checks: volume effect, time stability, day-of-week
sessions["vol_first_norm"] = sessions.groupby("ticker")["vol_first_hour"].transform(
    lambda x: x / x.rolling(20, min_periods=5).mean()
)

print("Correlation by first-hour volume bucket:")
sessions["vol_bucket"] = pd.qcut(sessions["vol_first_norm"].fillna(1), 5,
                                  labels=["Q1 (low vol)", "Q2", "Q3", "Q4", "Q5 (high vol)"],
                                  duplicates="drop")

for bucket, grp in sessions.groupby("vol_bucket"):
    if len(grp) < 30:
        continue
    c, p = stats.pearsonr(grp["first_hour_ret"], grp["rest_of_day_ret"])
    print(f"  {bucket}: corr = {c:+.4f} (p={p:.4f}, n={len(grp)})")

sorted_dates = sorted(sessions["date"].unique())
mid_date = sorted_dates[len(sorted_dates) // 2]
first_half = sessions[sessions["date"] < mid_date]
second_half = sessions[sessions["date"] >= mid_date]

print(f"\n--- Time stability ---")
print(f"First half ({first_half['date'].min()} - {first_half['date'].max()}):")
c1, p1 = stats.pearsonr(first_half["first_hour_ret"], first_half["rest_of_day_ret"])
print(f"  corr = {c1:+.4f} (p={p1:.4f}, n={len(first_half)})")

print(f"Second half ({second_half['date'].min()} - {second_half['date'].max()}):")
c2, p2 = stats.pearsonr(second_half["first_hour_ret"], second_half["rest_of_day_ret"])
print(f"  corr = {c2:+.4f} (p={p2:.4f}, n={len(second_half)})")

if np.sign(c1) == np.sign(c2):
    print(f"  Correlation sign consistent across halves")
else:
    print(f"  Correlation sign flips across halves - effect NOT stable")

sessions["dow"] = pd.to_datetime(sessions["date"]).dt.day_name()
print(f"\n--- Day of week ---")
dow_rows = []
for dow, grp in sessions.groupby("dow"):
    if len(grp) < 30:
        continue
    c, _ = stats.pearsonr(grp["first_hour_ret"], grp["rest_of_day_ret"])
    dow_rows.append({"dow": dow, "n": len(grp), "corr": c,
                     "mean_rod_pct": grp["rest_of_day_ret_pct"].mean()})
dow_stats = pd.DataFrame(dow_rows).set_index("dow").round(4)
day_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday"]
dow_stats = dow_stats.reindex([d for d in day_order if d in dow_stats.index])
print(dow_stats)

In [ ]:
# Part 2: Load extended universe (~100 stocks) + 10min candles + IMOEX index
from data_cache import get_stocks, get_index, RU_UNIVERSE_100, MOEX_INDEX

print(f"Universe: {len(RU_UNIVERSE_100)} tickers")

print("=== Hourly candles ===")
stocks_60 = get_stocks(RU_UNIVERSE_100, interval=60, start="2024-01-01", verbose=True)
print(f"\nTotal: {len(stocks_60)} candles, {stocks_60['ticker'].nunique()} tickers")

print("\n=== 10-minute candles ===")
stocks_10 = get_stocks(RU_UNIVERSE_100, interval=10, start="2024-01-01", verbose=True)
print(f"\nTotal: {len(stocks_10)} candles")

print("\n=== IMOEX index ===")
imoex = get_index(MOEX_INDEX, interval=60, start="2024-01-01", verbose=True)
print(f"Total: {len(imoex)} hourly IMOEX candles")

In [ ]:
# Retry missing 10-min data
MISSING_10M = ["ALRS", "PHOR", "RUAL", "TATNP"]
print("Retrying 10-minute candles...")
missing_data = get_stocks(MISSING_10M, interval=10, start="2024-01-01", verbose=True)
print(f"\nRetried: {len(missing_data)} candles")

stocks_10 = get_stocks(RU_UNIVERSE_100, interval=10, start="2024-01-01", verbose=False)
print(f"Total 10-min candles: {len(stocks_10)}, {stocks_10['ticker'].nunique()} tickers")

In [ ]:
# Retry missing 10-min data (attempt 2)
MISSING_10M = ["ALRS", "PHOR", "RUAL"]
print("Retrying 10-minute candles (attempt 2)...")
missing_data = get_stocks(MISSING_10M, interval=10, start="2024-01-01", verbose=True)
print(f"\nRetried: {len(missing_data)} candles")

In [ ]:
# Parameterized session extraction for different morning windows
def extract_sessions_window(df, window_minutes=60):
    df = df.copy()
    df["date"] = df["timestamp"].dt.date
    df["hour"] = df["timestamp"].dt.hour
    df["minute"] = df["timestamp"].dt.minute
    df["time_min"] = df["hour"] * 60 + df["minute"]
    main = df[(df["time_min"] >= 600) & (df["time_min"] <= 18*60 + 45)]
    window_end_min = 600 + window_minutes
    sessions = []
    for (ticker, d), grp in main.groupby(["ticker", "date"]):
        grp = grp.sort_values("timestamp")
        window_candles = grp[grp["time_min"] < window_end_min]
        if window_candles.empty:
            continue
        rest_candles = grp[grp["time_min"] >= window_end_min]
        if rest_candles.empty:
            continue
        open_morning = window_candles.iloc[0]["open"]
        close_window = window_candles.iloc[-1]["close"]
        vol_window = window_candles["volume_rub"].sum()
        close_eod = rest_candles.iloc[-1]["close"]
        morning_ret = (close_window - open_morning) / open_morning
        rest_ret = (close_eod - close_window) / close_window
        sessions.append({
            "ticker": ticker, "date": d,
            "open_morning": open_morning, "close_window": close_window,
            "close_eod": close_eod, "vol_window": vol_window,
            "morning_ret": morning_ret, "rest_of_day_ret": rest_ret,
        })
    return pd.DataFrame(sessions)


print("Extracting sessions for different windows...")
sessions_10  = extract_sessions_window(stocks_10, window_minutes=10)
print(f"  10 min: {len(sessions_10)} sessions")
sessions_30  = extract_sessions_window(stocks_10, window_minutes=30)
print(f"  30 min: {len(sessions_30)} sessions")
sessions_60_new = extract_sessions_window(stocks_60, window_minutes=60)
print(f"  60 min: {len(sessions_60_new)} sessions")

In [ ]:
# Beta-neutralization: subtract IMOEX return from stock returns
def add_market_neutral(sessions_df, index_df, window_minutes=60):
    idx = index_df.copy()
    idx["date"] = idx["timestamp"].dt.date
    idx["hour"] = idx["timestamp"].dt.hour
    idx["minute"] = idx["timestamp"].dt.minute
    idx["time_min"] = idx["hour"] * 60 + idx["minute"]
    idx = idx[(idx["time_min"] >= 600) & (idx["time_min"] <= 18*60 + 45)]
    window_end_min = 600 + window_minutes
    idx_per_day = {}
    for d, grp in idx.groupby("date"):
        grp = grp.sort_values("timestamp")
        win = grp[grp["time_min"] < window_end_min]
        rest = grp[grp["time_min"] >= window_end_min]
        if win.empty or rest.empty:
            continue
        open_m = win.iloc[0]["open"]
        close_w = win.iloc[-1]["close"]
        close_eod = rest.iloc[-1]["close"]
        idx_per_day[d] = {
            "idx_morning_ret": (close_w - open_m) / open_m,
            "idx_rest_ret": (close_eod - close_w) / close_w,
        }
    idx_df = pd.DataFrame.from_dict(idx_per_day, orient="index")
    idx_df.index.name = "date"
    out = sessions_df.merge(idx_df, on="date", how="left")
    out["alpha_morning"] = out["morning_ret"] - out["idx_morning_ret"]
    out["alpha_rest"]    = out["rest_of_day_ret"] - out["idx_rest_ret"]
    return out


sessions_10_mn  = add_market_neutral(sessions_10, imoex, window_minutes=10)
sessions_30_mn  = add_market_neutral(sessions_30, imoex, window_minutes=30)
sessions_60_mn  = add_market_neutral(sessions_60_new, imoex, window_minutes=60)

print(f"10min: {len(sessions_10_mn)} sessions, index matched: {sessions_10_mn['idx_rest_ret'].notna().sum()}")
print(f"30min: {len(sessions_30_mn)} sessions, index matched: {sessions_30_mn['idx_rest_ret'].notna().sum()}")
print(f"60min: {len(sessions_60_mn)} sessions, index matched: {sessions_60_mn['idx_rest_ret'].notna().sum()}")

for df in [sessions_10_mn, sessions_30_mn, sessions_60_mn]:
    df.dropna(subset=["alpha_morning", "alpha_rest"], inplace=True)

In [ ]:
# Summary table: correlations and quintile spreads across all windows
def window_summary(df, label, morning_col="morning_ret", rest_col="rest_of_day_ret"):
    df = df.dropna(subset=[morning_col, rest_col])
    if len(df) < 100:
        return None
    c, p = stats.pearsonr(df[morning_col], df[rest_col])
    df = df.copy()
    df["_q"] = pd.qcut(df[morning_col], 5, labels=False, duplicates="drop")
    q_stats = df.groupby("_q").agg(
        n=(rest_col, "count"),
        m_mean=(morning_col, "mean"),
        r_mean=(rest_col, "mean"),
        r_winrate=(rest_col, lambda x: (x > 0).mean()),
    )
    spread = q_stats.loc[4, "r_mean"] - q_stats.loc[0, "r_mean"]
    return {
        "label": label, "n": len(df), "corr": c, "p_value": p,
        "Q1_mean_pct": q_stats.loc[0, "r_mean"] * 100,
        "Q5_mean_pct": q_stats.loc[4, "r_mean"] * 100,
        "Q5-Q1_spread_pct": spread * 100,
        "Q1_winrate": q_stats.loc[0, "r_winrate"],
        "Q5_winrate": q_stats.loc[4, "r_winrate"],
    }


results = []
for window_label, sess_df in [("10min", sessions_10_mn),
                                ("30min", sessions_30_mn),
                                ("60min", sessions_60_mn)]:
    r = window_summary(sess_df, f"{window_label} raw", "morning_ret", "rest_of_day_ret")
    if r:
        results.append(r)
    r = window_summary(sess_df, f"{window_label} alpha (vs IMOEX)", "alpha_morning", "alpha_rest")
    if r:
        results.append(r)

results_df = pd.DataFrame(results)
print("=" * 80)
print("SUMMARY: correlation and Q5-Q1 spread across all windows")
print("=" * 80)
display(results_df.round(4))

In [ ]:
# Long-short backtest on extended universe with volume filter
for df in [sessions_10_mn, sessions_30_mn, sessions_60_mn]:
    df["vol_norm"] = df.groupby("ticker")["vol_window"].transform(
        lambda x: x / x.rolling(20, min_periods=5).mean()
    )


def backtest_long_short(df, top_n=5, morning_col="morning_ret",
                         rest_col="rest_of_day_ret",
                         min_morning_pct=None, min_vol_norm=None,
                         cost_pct=TOTAL_COST_PCT):
    trades = []
    for d, day in df.groupby("date"):
        if min_vol_norm is not None:
            day = day[day["vol_norm"].fillna(0) >= min_vol_norm]
        if min_morning_pct is not None:
            day = day[day[morning_col].abs() * 100 >= min_morning_pct]
        if len(day) < 2 * top_n:
            continue
        longs = day.nlargest(top_n, morning_col)
        shorts = day.nsmallest(top_n, morning_col)
        for _, r in longs.iterrows():
            pnl = r[rest_col] - cost_pct
            trades.append({"date": d, "ticker": r["ticker"], "side": "long",
                            "morning_ret": r[morning_col], "rest_ret": r[rest_col],
                            "pnl_net": pnl})
        for _, r in shorts.iterrows():
            pnl = -r[rest_col] - cost_pct
            trades.append({"date": d, "ticker": r["ticker"], "side": "short",
                            "morning_ret": r[morning_col], "rest_ret": r[rest_col],
                            "pnl_net": pnl})
    return pd.DataFrame(trades)


def quick_metrics(trades):
    if trades.empty:
        return None
    daily = trades.groupby("date")["pnl_net"].mean()
    sharpe = daily.mean() / daily.std() * np.sqrt(252) if daily.std() > 0 else 0
    return {
        "trades": len(trades), "days": len(daily),
        "winrate_pct": (trades["pnl_net"] > 0).mean() * 100,
        "avg_pnl_pct": trades["pnl_net"].mean() * 100,
        "annual_ret_pct": daily.mean() * 252 * 100,
        "sharpe": sharpe,
        "max_dd_pct": (daily.cumsum().cummax() - daily.cumsum()).max() * 100,
    }


bt_results = []
configs = [
    ("10m raw top5",         sessions_10_mn, "morning_ret",   "rest_of_day_ret", None, None),
    ("10m raw top5 vol2x",   sessions_10_mn, "morning_ret",   "rest_of_day_ret", None, 2.0),
    ("10m alpha top5",       sessions_10_mn, "alpha_morning", "alpha_rest",      None, None),
    ("10m alpha top5 vol2x", sessions_10_mn, "alpha_morning", "alpha_rest",      None, 2.0),
    ("30m raw top5",         sessions_30_mn, "morning_ret",   "rest_of_day_ret", None, None),
    ("30m alpha top5",       sessions_30_mn, "alpha_morning", "alpha_rest",      None, None),
    ("30m alpha top5 vol2x", sessions_30_mn, "alpha_morning", "alpha_rest",      None, 2.0),
    ("60m raw top5",         sessions_60_mn, "morning_ret",   "rest_of_day_ret", None, None),
    ("60m alpha top5",       sessions_60_mn, "alpha_morning", "alpha_rest",      None, None),
    ("60m alpha top5 vol2x", sessions_60_mn, "alpha_morning", "alpha_rest",      None, 2.0),
]

for cfg in configs:
    label, df, mcol, rcol, min_m, min_v = cfg
    trades = backtest_long_short(df, top_n=5, morning_col=mcol, rest_col=rcol,
                                   min_morning_pct=min_m, min_vol_norm=min_v)
    m = quick_metrics(trades)
    if m:
        m["label"] = label
        bt_results.append(m)

bt_df = pd.DataFrame(bt_results).sort_values("sharpe", ascending=False)
cols = ["label", "trades", "days", "winrate_pct", "avg_pnl_pct",
        "annual_ret_pct", "sharpe", "max_dd_pct"]
print("=" * 80)
print("LONG-SHORT STRATEGY (long top movers + short bottom movers)")
print("=" * 80)
display(bt_df[cols].round(3))

In [ ]:
# Equity curves for top-3 configurations
top3 = bt_df.head(3)

fig, axes = plt.subplots(3, 1, figsize=(14, 11))

for i, (_, row) in enumerate(top3.iterrows()):
    label = row["label"]
    cfg = next(c for c in configs if c[0] == label)
    _, df, mcol, rcol, min_m, min_v = cfg
    trades = backtest_long_short(df, top_n=5, morning_col=mcol, rest_col=rcol,
                                   min_morning_pct=min_m, min_vol_norm=min_v)
    if trades.empty:
        continue
    daily = trades.groupby("date")["pnl_net"].mean()
    equity = daily.cumsum() * 100
    color = "#2ecc71" if row["sharpe"] > 0 else "#e74c3c"
    axes[i].plot(pd.to_datetime(equity.index), equity, color=color, linewidth=1.5)
    axes[i].fill_between(pd.to_datetime(equity.index), equity, alpha=0.15, color=color)
    axes[i].axhline(0, color="black", linewidth=0.5)
    axes[i].set_title(f"#{i+1}: {label} | Sharpe {row['sharpe']:.2f} | "
                       f"Ann {row['annual_ret_pct']:.1f}% | MaxDD {row['max_dd_pct']:.1f}%")
    axes[i].set_ylabel("Cumulative return, %")

plt.tight_layout()
plt.show()

## Results

The intraday momentum hypothesis is dead. All configurations produce negative Sharpe ratios, with no evidence of continuation effect on MOEX stocks.

- **Part 1 (top-20 liquid stocks):** Near-zero correlation between first-hour and rest-of-day returns (r=+0.001, p=0.89). All backtest variants (long winners, short winners, long losers, various thresholds) yield deeply negative Sharpe (-3 to -14). Quintile analysis shows monotonically declining rest-of-day returns from Q1 to Q5.
- **Part 2 (100 stocks, beta-neutral, volume filters):** Expanded universe confirms reversal, not continuation. All window lengths (10/30/60 min) show significant negative correlation, especially after beta-neutralization (r=-0.17 for 10min alpha). Long-short backtests across all configurations produce Sharpe between -7 and -14. Volume filters do not help.
- **Conclusion:** No exploitable intraday momentum on MOEX. The weak reversal signal exists but does not survive transaction costs.